### Оптимизаторы, параметры и модули

Продолжаем работать с логистической регрессией. В прошлом ноутбуке мы обновляли параметры вручную — теперь научимся делегировать эту работу Pytorch.

In [1]:
import torch

In [2]:
torch.manual_seed(42)
n_samples = 16
n_features = 5
x = torch.randn(n_samples, n_features)  # входной тензор
y = torch.randint(2, size=(n_samples, 1)).float()  # выходной тензор
w = torch.randn(
    n_features, 1, requires_grad=True
)  # параметр, хотим обновлять градиентным спуском
b = torch.randn(1, requires_grad=True)  # параметр, хотим обновлять градиентным спуском

In [3]:
loss = torch.nn.functional.binary_cross_entropy_with_logits(x @ w + b, y)
print(f"Начальное значение ошибки: {loss:.4f}")

Начальное значение ошибки: 0.8950


#### 1. Добавляем оптимизатор

Вместо того чтобы вручную обновлять параметры внутри `torch.no_grad()`, передадим их оптимизатору. Он возьмёт на себя и обновление (`optimizer.step()`), и обнуление градиентов (`optimizer.zero_grad()`):

In [4]:
optimizer = torch.optim.SGD([w, b], lr=0.1)

In [5]:
n_iter = 1000
for i in range(n_iter):
    z = x @ w + b
    loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print(f"Финальное значение ошибки: {loss:.4f}")

Финальное значение ошибки: 0.3861


#### 2. Прячем параметры модели внутрь `torch.nn.Module`

Когда параметров много, передавать их списком неудобно. `nn.Module` — базовый класс для всех моделей в Pytorch. Он хранит параметры и предоставляет метод `.parameters()` для передачи их оптимизатору.

In [6]:
class LogReg(torch.nn.Module):
    def __init__(self, in_features: int, n_classes: int) -> None:
        super().__init__()
        # nn.Parameter — обёртка над тензором, которая регистрирует его как параметр модели,
        # чтобы .parameters() мог его найти и передать оптимизатору
        self.weight = torch.nn.Parameter(torch.randn(in_features, n_classes))
        self.bias = torch.nn.Parameter(torch.randn(n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

In [7]:
torch.manual_seed(42)
n_samples = 16
n_features = 5
x = torch.randn(n_samples, n_features)  # входной тензор
y = torch.randint(2, size=(n_samples, 1)).float()  # выходной тензор

logreg = LogReg(n_features, 1)
optimizer = torch.optim.SGD(logreg.parameters(), lr=0.1)

In [8]:
n_iter = 1000
for i in range(n_iter):
    z = logreg(x)  # эквивалентно logreg.forward(x), но вызывает __call__
    loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print(f"Финальное значение ошибки: {loss:.4f}")

Финальное значение ошибки: 0.3861


#### 3. Как учить на GPU

Данные и модель должны быть на одном устройстве. Метод `.to(device)` переносит тензоры и параметры модели на нужное устройство.

In [ ]:
# 'cuda' для NVIDIA GPU, 'mps' для Mac M1-4, 'cpu' если нет GPU
accelerator = "mps"
print(x.device)
x = x.to(device=accelerator)
print(x.device)

In [10]:
x = x.to(device=accelerator)
y = y.to(device=accelerator)
logreg.to(device=accelerator)

for i in range(n_iter):
    z = logreg(x)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print(f"Финальное значение ошибки: {loss:.4f}")

Финальное значение ошибки: 0.3833


#### 4. Как сохранять и загружать веса модели

Рекомендуемый способ: сохраняем только веса через `state_dict()`, а при загрузке создаём модель заново.

In [11]:
# сохраняем состояние модуля
torch.save(logreg.state_dict(), "logreg.pt")

# загружаем: создаём модель с той же архитектурой и подгружаем веса
new_model = LogReg(n_features, 1)
new_model.load_state_dict(torch.load("logreg.pt", weights_only=True))
z = new_model(x.to(device="cpu"))
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y.to(device="cpu"))
print(f"Ошибка загруженной модели: {loss:.4f}")

Ошибка загруженной модели: 0.3833


Также можно сохранить модель целиком через `torch.save(model, path)` — это использует `pickle` и менее надёжно, поэтому предпочитайте `state_dict`.

#### 5. Готовые слои: `nn.Linear`

Писать `nn.Parameter(torch.randn(...))` каждый раз утомительно. В Pytorch уже есть готовые модули для типовых операций. Линейный слой реализован в `torch.nn.Linear`:

In [12]:
linear = torch.nn.Linear(in_features=5, out_features=1)

# внутри — те же Parameter, что мы создавали вручную
print(linear.weight.shape)  # (out_features, in_features)
print(linear.bias.shape)    # (out_features,)

torch.Size([1, 5])
torch.Size([1])


Наша логистическая регрессия в одну строчку:

In [13]:
class LogRegV2(torch.nn.Module):
    def __init__(self, in_features: int, n_classes: int) -> None:
        super().__init__()
        self.linear = torch.nn.Linear(in_features, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)